<a href="https://colab.research.google.com/github/kofisarf/Plant-Disease-Detection-Group-9/blob/main/Plant_Disease_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"kofiboadisarfo","key":"1fdca5f8b8c4444d8c10244e4b29ac88"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_data

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:21<00:00, 99.6MB/s]



In [5]:
import os
import shutil

source_dir = 'plantvillage_data/plantvillage dataset/color'

Crop_data = 'Crop_data'

os.makedirs(Crop_data, exist_ok=True)

Study_crops = ['Tomato', 'Corn_(maize)', 'Potato', 'Pepper']

for folder_name in os.listdir(source_dir):
    if any(crop in folder_name for crop in Study_crops):

        src_path = os.path.join(source_dir, folder_name)
        dst_path = os.path.join(Crop_data, folder_name)

        if not os.path.exists(dst_path):
            shutil.copytree(src_path, dst_path)
            print(f"Copied: {folder_name}")

print("Data filtering complete!")

Copied: Pepper,_bell___healthy
Copied: Tomato___Septoria_leaf_spot
Copied: Tomato___Tomato_Yellow_Leaf_Curl_Virus
Copied: Corn_(maize)___Northern_Leaf_Blight
Copied: Potato___Late_blight
Copied: Corn_(maize)___Common_rust_
Copied: Tomato___healthy
Copied: Corn_(maize)___healthy
Copied: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Copied: Tomato___Bacterial_spot
Copied: Potato___healthy
Copied: Tomato___Tomato_mosaic_virus
Copied: Tomato___Target_Spot
Copied: Potato___Early_blight
Copied: Tomato___Late_blight
Copied: Pepper,_bell___Bacterial_spot
Copied: Tomato___Spider_mites Two-spotted_spider_mite
Copied: Tomato___Early_blight
Copied: Tomato___Leaf_Mold
Data filtering complete!


In [6]:
# split-folders helper
!pip install split-folders

import splitfolders

# Split your filtered dataset into train (80%), val (10%), test (10%)
splitfolders.ratio(
    "Crop_data",       # Your filtered folder name
    output="data_split",      # New folder with split subdirectories
    seed=42,
    ratio=(0.8, 0.1, 0.1)
)

print("✅ Data successfully split into train, val, and test folders!")

Copying files: 26639 files [00:06, 4227.93 files/s]

✅ Data successfully split into train, val, and test folders!


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Rescale image pixels (from 0-255 down to 0.0-1.0)
datagen = ImageDataGenerator(rescale=1./255)

IMG_SIZE = (128, 128) # Resizing images to 128x128 keeps training fast for Thursday's demo
BATCH_SIZE = 32

# Create the training stream
train_generator = datagen.flow_from_directory(
    'data_split/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Create the validation stream
val_generator = datagen.flow_from_directory(
    'data_split/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Create the testing stream
test_generator = datagen.flow_from_directory(
    'data_split/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 21303 images belonging to 19 classes.
Found 2657 images belonging to 19 classes.
Found 2679 images belonging to 19 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Get the number of crop classes automatically
num_classes = train_generator.num_classes

# Build a simple baseline CNN architecture
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax') # Outputs probability for each crop disease class
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model for 5 epochs to establish a quick baseline
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
137/666 ━━━━━━━━━━━━━━━━━━━━ 8:27 959ms/step - accuracy: 0.3274 - loss: 2.3119